<a href="https://colab.research.google.com/github/arjunnamburi/ml_death_overs_ipl/blob/main/notebooks/models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/IPL PROJECT ALL DOCS/data/'
SEED = 42                      # fixed seed, so the bootstrap results are reproducible

df = pd.read_parquet(DATA + 'ipl_features.parquet')
with open(DATA + 'feature_list.json') as f:
    meta = json.load(f)
FEATURES, CAT_FEATURES, CLASSES = meta['features'], meta['categorical'], meta['classes']
K = len(CLASSES)               # 7 classes: 0, 1, 2, 3, 4, 6, Wicket

# Legal balls in train and validation only
d = df[(df['y'] >= 0) & df['split'].isin(['train', 'val'])].copy()
d['y'] = d['y'].astype(int)

TR = (d['split'] == 'train').values
VA = (d['split'] == 'val').values
DEATH = (d['match_phase'] == 'death').values

Mounted at /content/drive


In [3]:
print('1.', len(df), len(d))                                   # all deliveries vs our working rows
print('2.', d['split'].value_counts().to_dict())              # Expected: train 244269, val 15809
print('3.', sorted(d['innings'].unique()))                     # Expected: [1, 2]
print('4.', (TR & DEATH).sum(), (VA & DEATH).sum())            # Expected: 55634 3466
print('5.', d['match_phase'].unique().tolist())                # Expected: powerplay, middle, death
print('6.', CLASSES, K)                                        # Expected: ['0','1','2','3','4','6','Wicket'] 7
print('7.', d['y'].value_counts(normalize=True).sort_index().round(3).tolist())
print('8.', d['rain_affected'].sum())                          # Expected: 0

1. 295553 260078
2. {'train': 244269, 'val': 15809}
3. [np.int64(1), np.int64(2)]
4. 55634 3466
5. ['powerplay', 'middle', 'death']
6. ['0', '1', '2', '3', '4', '6', 'Wicket'] 7
7. [0.308, 0.398, 0.066, 0.003, 0.121, 0.053, 0.051]
8. 0


In [4]:
EPS = 1e-15
LOSS = {}       # per-ball log-losses, keyed by (model, split), for paired comparisons later
results = []    # the results table: one row per (model, split, slice), grows with every model

def per_ball_logloss(P, y):
    """-ln(probability given to the outcome that actually happened), one value per ball."""
    p_true = P[np.arange(len(y)), y]
    return -np.log(np.clip(p_true, EPS, 1))

def per_ball_brier(P, y):
    """Sum over the 7 classes of (p_c - o_c)^2, one value per ball."""
    O = np.zeros_like(P)
    O[np.arange(len(y)), y] = 1
    return ((P - O) ** 2).sum(axis=1)

def match_bootstrap(values, match_ids, B=2000, seed=SEED):
    """95% interval for the mean of per-ball values, resampling whole matches."""
    s = pd.DataFrame({'m': match_ids, 'v': values}).groupby('m')['v'].agg(['sum', 'count'])
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(s), size=(B, len(s)))          # B resamples, each a list of match positions
    means = s['sum'].values[idx].sum(axis=1) / s['count'].values[idx].sum(axis=1)
    return np.percentile(means, [2.5, 97.5])

def evaluate(name, P, rows, split_name, slices=('all', 'death')):
    """Score probabilities P (one row per ball in `rows`), keep the per-ball losses, add rows to the results table."""
    assert P.shape == (rows.sum(), K), 'one row of 7 probabilities per ball'
    assert np.allclose(P.sum(axis=1), 1), 'probabilities must sum to 1'
    assert (P > 0).all(), 'a probability of 0 makes log-loss infinite'
    y = d.loc[rows, 'y'].values
    m = d.loc[rows, 'match_id'].values
    death = DEATH[rows]
    ll, br = per_ball_logloss(P, y), per_ball_brier(P, y)
    LOSS[name, split_name] = pd.Series(ll, index=d.index[rows])
    results[:] = [r for r in results if (r['model'], r['split']) != (name, split_name)]   # rerun-safe
    for sl in slices:
        mask = death if sl == 'death' else np.ones(len(y), bool)
        lo, hi = match_bootstrap(ll[mask], m[mask])
        results.append(dict(model=name, split=split_name, slice=sl, balls=int(mask.sum()),
                            logloss=ll[mask].mean(), ll_low=lo, ll_high=hi, brier=br[mask].mean()))
    return pd.DataFrame(results).query('model == @name and split == @split_name').round(4)

def compare(a, b, split_name, sl='all'):
    """Mean log-loss of model a minus model b on the same balls, with a paired match-bootstrap interval.
    Negative means a is better."""
    diff = (LOSS[a, split_name] - LOSS[b, split_name]).dropna()    # only balls both models scored
    if sl == 'death':
        diff = diff[(d.loc[diff.index, 'match_phase'] == 'death').values]
    lo, hi = match_bootstrap(diff.values, d.loc[diff.index, 'match_id'].values)
    return pd.Series({'diff': diff.mean(), 'low': lo, 'high': hi, 'balls': len(diff)}).round(4)

def loss_by_class(name, split_name):
    """Which outcomes the log-loss comes from: share of balls, average loss, share of total loss."""
    ll = LOSS[name, split_name]
    t = pd.DataFrame({'cls': d.loc[ll.index, 'y'].values, 'll': ll.values}).groupby('cls')['ll'].agg(['count', 'mean', 'sum'])
    t.index = [CLASSES[i] for i in t.index]
    return pd.DataFrame({'share_of_balls': t['count'] / t['count'].sum(),
                         'avg_loss': t['mean'],
                         'share_of_loss': t['sum'] / t['sum'].sum()}).round(3)

In [5]:
yv = d.loc[VA, 'y'].values
U = np.full((VA.sum(), K), 1 / K)                     # uniform: 1/7 on every class
print('1.', per_ball_logloss(U, yv).mean().round(4), np.log(K).round(4))     # Expected: 1.9459 1.9459
print('2.', per_ball_brier(U, yv).mean().round(4), round(6 / 7, 4))         # Expected: 0.8571 0.8571
Perfect = np.full((VA.sum(), K), 1e-9); Perfect[np.arange(len(yv)), yv] = 1 - 6e-9
print('3.', per_ball_logloss(Perfect, yv).mean().round(4), per_ball_brier(Perfect, yv).mean().round(4))   # Expected: 0.0 0.0
evaluate('Uniform', U, VA, 'val')

1. 1.9459 1.9459
2. 0.8571 0.8571
3. 0.0 0.0


,model,split,slice,balls,logloss,ll_low,ll_high,brier
0,Uniform,val,all,15809,1.9459,1.9459,1.9459,0.8571
1,Uniform,val,death,3466,1.9459,1.9459,1.9459,0.8571


In [6]:
def fit_freq(y):
    """Share of each of the 7 classes in y."""
    return np.bincount(y, minlength=K) / len(y)

def predict_const(p, rows):
    """The same 7 probabilities for every ball in rows."""
    return np.tile(p, (rows.sum(), 1))

# B0: training class frequencies, predicted for every ball
p_b0 = fit_freq(d.loc[TR, 'y'].values)
print(pd.Series(p_b0, index=CLASSES).round(4))

evaluate('B0', predict_const(p_b0, TR), TR, 'train')
evaluate('B0', predict_const(p_b0, VA), VA, 'val')

# B0-death: frequencies from training death-over balls only, scored on validation death balls
p_b0d = fit_freq(d.loc[TR & DEATH, 'y'].values)
evaluate('B0-death', predict_const(p_b0d, VA & DEATH), VA & DEATH, 'val', slices=('death',))

pd.DataFrame(results).round(4)

0         0.3102
1         0.3982
2         0.0660
3         0.0032
4         0.1202
6         0.0513
Wicket    0.0510
dtype: float64


,model,split,slice,balls,logloss,ll_low,ll_high,brier
0,Uniform,val,all,15809,1.9459,1.9459,1.9459,0.8571
1,Uniform,val,death,3466,1.9459,1.9459,1.9459,0.8571
2,B0,train,all,244269,1.4862,1.4823,1.4901,0.7212
3,B0,train,death,55634,1.6378,1.6297,1.6457,0.7649
4,B0,val,all,15809,1.5391,1.5229,1.5557,0.7391
5,B0,val,death,3466,1.6682,1.6388,1.6985,0.7754
6,B0-death,val,death,3466,1.6124,1.5868,1.6378,0.7579


In [7]:
H_train = -(p_b0 * np.log(p_b0)).sum()
print('1.', p_b0.sum().round(6))                                       # Expected: 1.0
print('2.', H_train.round(4), LOSS['B0', 'train'].mean().round(4))     # Expected: equal, about 1.485
print('3.'); print(compare('B0-death', 'B0', 'val', 'death'))          # Expected: diff negative
print('4.'); print(loss_by_class('B0', 'val'))

1. 1.0
2. 1.4862 1.4862
3.
diff       -0.0558
low        -0.0652
high       -0.0458
balls    3466.0000
dtype: float64
4.
        share_of_balls  avg_loss  share_of_loss
0                0.270     1.171          0.205
1                0.399     0.921          0.239
2                0.059     2.718          0.105
3                0.002     5.749          0.006
4                0.139     2.119          0.191
6                0.079     2.970          0.153
Wicket           0.052     2.977          0.101
